# WWR Segmentation — Google Colab (A100)

### Storage (Google Drive)
| Location | Use |
|----------|-----|
| `/content/data` | train images (fast local SSD, from `data.zip`) |
| `My Drive/WWR_Seg_Model/` | `code.zip`, `data.zip`, **best model**, logs, results |

> Only **one** `best_model.keras` (~630 MB) is saved — not one file per epoch.
> Delete old `training_*.keras` files on Drive to free space before training.

### Training monitor
Each epoch shows **Input | Ground Truth | Prediction** inline in Colab.
Saved PNGs: `WWR_Seg_Model/results/epoch_visualizations/training/`

### In Colab
1. **Runtime → A100 GPU**
2. Run all cells in order

In [ ]:
# Install dependencies
!pip install -q tensorflow>=2.15 numpy pandas matplotlib scikit-learn openpyxl

## 1. Mount Drive & Load Project Code (required — run before any import)

In [ ]:
import os
import sys
import shutil
import zipfile
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

LOCAL_REPO = Path("/content/U-Net_Segmentation")
DRIVE_BASE = Path("/content/drive/MyDrive/WWR_Seg_Model")
CODE_ZIP = DRIVE_BASE / "code.zip"
ALLOW_UPLOAD = True  # set False if code.zip is already on Drive

# ── Mount Google Drive ───────────────────────────────────────────────────────
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")


def _has_pkg(root: Path) -> bool:
    return (root / "WWR_Segmentation" / "__init__.py").exists()


def _find_on_drive() -> Path | None:
    if not DRIVE_BASE.exists():
        return None
    for c in [DRIVE_BASE / "code", DRIVE_BASE / "U-Net_Segmentation", DRIVE_BASE]:
        if _has_pkg(c):
            return c.resolve()
    for f in DRIVE_BASE.rglob("WWR_Segmentation/__init__.py"):
        return f.parent.parent.resolve()
    return None


def _extract_zip(zip_path: Path, target: Path = LOCAL_REPO) -> Path:
    tmp = Path("/content/_zip_extract")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir()
    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(tmp)
    root = None
    if _has_pkg(tmp):
        root = tmp
    else:
        for name in ("U-Net_Segmentation", "code", "U-Net_Segmentation-main"):
            if _has_pkg(tmp / name):
                root = tmp / name
                break
        if root is None:
            for f in tmp.rglob("WWR_Segmentation/__init__.py"):
                root = f.parent.parent
                break
    if root is None:
        raise FileNotFoundError(f"WWR_Segmentation not inside {zip_path.name}")
    if target.exists():
        shutil.rmtree(target)
    shutil.move(str(root), str(target))
    shutil.rmtree(tmp, ignore_errors=True)
    return target


def _copy_local(src: Path) -> Path:
    if src.resolve() == LOCAL_REPO.resolve():
        return LOCAL_REPO
    print(f"Copying to local SSD: {LOCAL_REPO}")
    if LOCAL_REPO.exists():
        shutil.rmtree(LOCAL_REPO)
    shutil.copytree(src, LOCAL_REPO, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".git"))
    return LOCAL_REPO


# ── Load project (Drive only) ────────────────────────────────────────────────
project_root = _find_on_drive()

if project_root is None and CODE_ZIP.exists():
    project_root = _extract_zip(CODE_ZIP)

if project_root is None and ALLOW_UPLOAD:
    from google.colab import files
    print("code.zip not on Drive — please upload it now:")
    up = files.upload()
    zpath = Path(f"/content/{next(iter(up))}")
    zpath.write_bytes(up[next(iter(up))])
    project_root = _extract_zip(zpath)

if project_root is None:
    raise FileNotFoundError(
        "Project code not found.\n\n"
        "1) On PC:  python scripts/pack_for_colab.py\n"
        "2) Upload code.zip → My Drive/WWR_Seg_Model/code.zip\n"
        "3) Re-run this cell"
    )

if str(project_root).startswith("/content/drive"):
    project_root = _copy_local(project_root)

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root : {project_root}")
print(f"Package OK   : {_has_pkg(project_root)}")

## 2. Setup — Unzip data & prepare training

Data is copied to local SSD for speed. Model and logs save to Google Drive (`WWR_Seg_Model/models/best_model.keras`).

In [ ]:
from WWR_Segmentation.colab_setup import setup_colab, CODE_VERSION
from WWR_Segmentation.dataset import get_dataset_info

print(f"Code version: {CODE_VERSION}")  # must be 2026.06.27.5

# force_unzip=True only if data.zip changed or first run
config = setup_colab(force_unzip=False)

info = get_dataset_info(config)
print(f"Best model → {config.best_model_path}")
print(f"Train: {info['train']} | Val: {info['val']} | Test: {info['test']}")

## 3. Training

In [ ]:
from WWR_Segmentation.trainer import train

model, history = train(config)

## 4. Evaluation (Validation — test set optional)

In [ ]:
from WWR_Segmentation.evaluate import run_evaluation
from WWR_Segmentation.dataset import has_test_set

eval_results = run_evaluation(config)
print(f"Val mean IoU: {eval_results['validation']['metrics']['mean_iou']['value']:.4f}")

if has_test_set(config) and "test" in eval_results:
    print(f"Test mean IoU: {eval_results['test']['metrics']['mean_iou']['value']:.4f}")
else:
    print("Test set not loaded yet — validation metrics above are sufficient for now.")

## 5. Window-to-Wall Ratio (WWR) — after test set is added

In [ ]:
# Run when the 67-image test set is on Drive
# from WWR_Segmentation.wwr import run_wwr_analysis
# from WWR_Segmentation.dataset import has_test_set
#
# if has_test_set(config):
#     wwr_df = run_wwr_analysis(config)
#     display(wwr_df.head())
# else:
#     print("Upload test set first, then re-run setup_colab(force_unzip=True)")

## 6. Optional: 5-Fold Cross-Validation

In [ ]:
# Uncomment to run (train set only — test set is never touched)
# from WWR_Segmentation.cross_validation import run_cross_validation
# cv_summary = run_cross_validation(config)

## 7. Inference — after test set is added

In [ ]:
# from WWR_Segmentation.inference import run_inference
# from WWR_Segmentation.dataset import has_test_set
#
# if has_test_set(config):
#     run_inference(config, input_dir=config.test_images_dir)
# else:
#     print("Upload test set first.")